<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_distribuido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 7: Conceptos de almacenamiento para Bigdata

## ***Universidad Central***
>## **Facultad de Ingeniería y Ciencias Básicas.**
>## ***Maestría en analítica de datos***
![Imágen1](https://universidad.ucentral.edu.co/tulengua/wp-content/themes/tulengua/images/logo-ucentral.png)


>## ***Big Data.***
>## ***Docente: Antonino Zainea Maya.***

## Objetivos de la Sesión:
* Por qué una sola máquina no escala

* Cómo funcionan los clusters

* Qué es fragmentación (sharding)

* Qué es replicación

*  Cómo se combinan ambas

*  Qué errores conceptuales rompen sistemas

---

# Escenario y contexto:
 Imagina que hace unos meses eras el único Chef de StatBurgers, manejando los pedidos desde tu laptop personal. Todo cabía en una hoja de Excel. Pero hoy, gracias a un video viral, StatBurgers es un fenómeno mundial: ¡tienes millones de pedidos por segundo entrando desde Bogotá hasta Tokio!
.

Para que el negocio no colapse, lanzamos la pregunta fundamental de la infraestructura moderna:

**¿Qué pasa si intento guardar todo (pedidos, inventario, clientes, nómina) en mi computador?**




**tu "restaurante" está a punto de quebrar técnicamente:**

* El Cuello de Botella (Performance): En un sistema centralizado, todas las consultas deben pasar por la computadora central para ser procesadas

*  Con millones de pedidos, tu procesador se saturará intentando responder a cada cliente, lo que aumentará drásticamente el tiempo de respuesta hasta hacer la aplicación inusable

* Punto Único de Falla (Reliability): Si tu computador se apaga, se daña el disco duro o se cae el internet de tu casa, todo el sistema de StatBurgers se detiene
* No hay respaldo; el negocio deja de existir hasta que repares esa máquina

* Límites de Almacenamiento: Una sola máquina tiene una capacidad física limitada de disco y memoria
* No importa qué tan potente sea, no podrá albergar el histórico de millones de transacciones por segundo que genera una operación global

* Costos de Escalabilidad: Intentar mejorar una sola máquina (escalabilidad vertical) se vuelve exponencialmente caro y llega a un tope físico donde ya no existen procesadores más rápidos para comprar.

In [ ]:
import numpy as np

print("👨‍🍳 Chef: 'No se preocupen, yo puedo procesar todos los tickets globales en mi laptop...'")

try:
    # Intentamos crear una matriz de 100,000 x 100,000 tickets de venta.
    # Cada número representa un monto en dólares (float64 = 8 bytes).
    # Tamaño estimado: (10^5 * 10^5 * 8 bytes) / 1024^3 ≈ 74.5 Gigabytes de RAM.

    # INICIA TU CODIGO AQUI
    mesa_del_chef = np.ones((100000, 100000), dtype=np.float64)
    # TERMINA TU CODIGO AQUI

    print("✅ ¡Sorpresa! Tu computadora es una supercomputadora de la NASA.")
except MemoryError:
    print("\n" + "!"*50)
    print("💥 >>> ERROR FATAL DE MEMORIA (MemoryError) <<< 💥")
    print("La mesa del Chef se rompió por el peso. El sistema ha colapsado.")
    print("!"*50)

👨‍🍳 Chef: 'No se preocupen, yo puedo procesar todos los tickets globales en mi laptop...'


### ATENCION: IMPORTANTE ANTES DE SEGUIR
Acabamos de saturar intencionalmente la memoria. Si intentas correr el resto de las celdas ahora mismo, pueden fallar.

**HAZ ESTO AHORA:**
Ve al menu superior de Colab -> **Entorno de ejecucion** -> **Reiniciar entorno de ejecucion**.
Ccontinua a partir de aqui abajo.



---

# Parte 1: Anatomía de un Ingrediente

Antes de que nuestra mesa colapse, el Chef debe entender por qué su "calculadora" de Python gasta tanta memoria. No es lo mismo anotar un número en una servilleta que mandarlo a imprimir en un libro de tapa dura con lomo de cuero.

### 1. La unidad de medida: Bits y Bytes

Para que el Chef se comunique con la cocina (el hardware), debe hablar en binario:

*   **Bit (Binary Digit):** La unidad mínima. Un `0` o un `1`. Es como un interruptor de luz.
*   **Byte:** Un grupo de **8 bits**. (Ejemplo: `01010000`). Es la unidad estándar de almacenamiento.
    *   *Corrección técnica:* Un byte **siempre** son 8 bits. Si tienes 8 interruptores, tienes 1 Byte.

### 2. ¿Por qué Python es tan "pesado"?

En lenguajes como **C**, un número entero ocupa solo **4 bytes**. Pero en **Python**, ese mismo número ocupa **28 bytes**.

**¿Por qué?** Porque en Python todo es un **OBJETO**.
Cuando creas un número, Python no solo guarda el valor; guarda un "paquete" que incluye:
1.  **Referencia:** Cuántas variables usan este número.
2.  **Tipo:** Una etiqueta que dice "Soy un entero".
3.  **Valor:** El número real.

Es la diferencia entre tener el ingrediente suelto (C) y tener el ingrediente en un frasco etiquetado con marca, fecha de vencimiento y manual de uso (Python).

---


In [ ]:
# Definimos los ingredientes
import sys
ticket_id = 1              # Entero (int)
monto = 150.50             # Decimal (float)
vacio = ""                 # String vacío (str)
lista_vacia = []           # Lista vacía (list)

print(f"--- PESO DE OBJETOS INDIVIDUALES ---")
print(f"ID del Ticket (int):    {sys.getsizeof(ticket_id)} bytes")
print(f"Monto de Venta (float): {sys.getsizeof(monto)} bytes")
print(f"Texto vacío (str):      {sys.getsizeof(vacio)} bytes")
print(f"Lista vacía (list):     {sys.getsizeof(lista_vacia)} bytes")


--- PESO DE OBJETOS INDIVIDUALES ---
ID del Ticket (int):    28 bytes
Monto de Venta (float): 24 bytes
Texto vacío (str):      41 bytes
Lista vacía (list):     56 bytes


In [ ]:
# Dato curioso: Un entero grande pesa más
print(f"Float gigante:        {sys.getsizeof(monto*100)} bytes")

Float gigante:        24 bytes


In [ ]:
# Dato curioso: Un entero grande pesa más
entero_gigante = 2**100
print(f"Entero gigante:        {sys.getsizeof(entero_gigante)} bytes")

Entero gigante:        40 bytes



###  El costo de las Listas (Punteros)

Las listas en Python no guardan los objetos *dentro* de la lista, sino **direcciones de memoria** (punteros). Cada dirección ocupa **8 bytes**.


In [ ]:
mi_lista = []
print(mi_lista)
print(f"Lista con 0 elementos: {sys.getsizeof(mi_lista)} bytes")

mi_lista.append(150.50)
print(mi_lista)
print(f"Lista con 1 elemento:  {sys.getsizeof(mi_lista)} bytes (Creció por el puntero)")

mi_lista.append(200.00)
mi_lista.append(3)
print(mi_lista)
print(f"Lista con 2 elementos: {sys.getsizeof(mi_lista)} bytes")

[]
Lista con 0 elementos: 56 bytes
[150.5]
Lista con 1 elemento:  88 bytes (Creció por el puntero)
[150.5, 200.0, 3]
Lista con 2 elementos: 88 bytes


In [ ]:
mi_lista = []
for i in range(10):
    mi_lista.append(i)
    print("Esta es la lista: ",mi_lista)
    print(f"Esta lista tiene {  sys.getsizeof(mi_lista)} bytes")

Esta es la lista:  [0]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2, 3]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2, 3, 4]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
Esta lista tiene 184 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Esta lista tiene 184 bytes



1.  **Lista con [0]: 88 bytes**
    *   El "envase" de la lista vacío pesa **56 o 64 bytes**.
    *   Python vio que metiste un elemento y dijo: *"Seguro vienen más, voy a reservar espacio para 4 elementos de una vez"*.
    *   **Matemática:** 56 (base) + (4 espacios × 8 bytes) = **88 bytes**.
    *   *Por eso, del elemento 1 al 4, el tamaño no cambia. Ya tenías el espacio pagado.*

2.  **Lista con [0, 1, 2, 3, 4]: 120 bytes**
    *   Al intentar meter el 5to elemento, la "cajita de 4" se llenó.
    *   Python pide una caja nueva más grande. En este caso, sumó espacio para **4 elementos más**.
    *   **Matemática:** 88 + (4 espacios × 8 bytes) = **120 bytes**.

3.  **Lista con 9 elementos: 184 bytes**
    *   La caja anterior se volvió a llenar. Python ahora pide un salto más grande (pide espacio para **8 elementos más**).
    *   **Matemática:** 120 + (8 espacios × 8 bytes) = **184 bytes**.

---

**¿Qué hace Python?**
Cuando le pides espacio para el primer huevo, Python no compra una cajita para uno. Compra una caja con espacio para **4 o 8 huevos** de una vez, previendo que vas a seguir agregando más.

A esto se le llama **Overallocation**: Python pide a la RAM más espacio del que necesita en ese momento para ganar **velocidad**. Es más rápido tener espacio vacío "por si acaso" que pedirle permiso a la memoria cada vez que haces un `.append()`.


Esta es la comparativa definitiva y técnicamente precisa para tu taller. Vamos a separar el **"Contenedor"** (la lista con sus punteros) del **"Contenido"** (los objetos numéricos), aplicando las reglas de **Overallocation** que descubriste.

---

### 🧮 Auditoría de Memoria: El Gran Almacén de StatBurgers

Para estos cálculos, usaremos los pesos de **CPython 64-bit**:
*   **Puntero (Dirección):** 8 bytes.
*   **Objeto Float:** 24 bytes.
*   **Objeto Int:** 28 bytes.
*   **Base de Lista:** 56 bytes.

---

### Escenario A: ¿Cuántas micro-listas (5 pedidos) caben en 1 GB?

Aquí el costo es alto porque creamos muchas "cajas" pequeñas. Python, al detectar que metes 5 elementos, reserva espacio para **8** (Overallocation inicial).

#### **Caso 1: Usando FLOATS (Decimales)**
1.  **La Lista (Contenedor):** 56 (base) + (8 espacios × 8 bytes) = **120 bytes**.
2.  **Los Números (Contenido):** 5 objetos × 24 bytes = **120 bytes**.
3.  **Total por lista:** **240 bytes**.
4.  **En 1 GB (mil millones de bytes):** $1.000.000.000 / 240 \approx$ **4.16 millones de listas.**

#### **Caso 2: Usando INTEGERS (Enteros)**
1.  **La Lista (Contenedor):** 56 (base) + (8 espacios × 8 bytes) = **120 bytes**.
2.  **Los Números (Contenido):** 5 objetos × 28 bytes = **140 bytes**.
3.  **Total por lista:** **260 bytes**.
4.  **En 1 GB:** $1.000.000.000 / 260 \approx$ **3.84 millones de listas.**

> **Conclusión A:** ¡Usar enteros para los IDs de los tickets gasta un **8% más de RAM** que usar floats para los precios!

---

### Escenario B: Una sola lista masiva (10.000 millones de pedidos)

En listas gigantes, la **Overallocation** de Python es de aproximadamente un **12.5%** extra para no quedarse sin espacio.

#### **Caso 1: Usando FLOATS**
1.  **Punteros (Raw):** 10.000 M × 8 bytes = **80 GB**.
2.  **Overallocation (Margen):** ~10.000 M × 1 byte (aprox 12.5%) = **10 GB**.
3.  **Objetos Float:** 10.000 M × 24 bytes = **240 GB**.
4.  **TOTAL REAL:** **330 GB de RAM.**

#### **Caso 2: Usando INTEGERS**
1.  **Punteros (Raw):** 10.000 M × 8 bytes = **80 GB**.
2.  **Overallocation (Margen):** **10 GB**.
3.  **Objetos Int:** 10.000 M × 28 bytes = **280 GB**.
4.  **TOTAL REAL:** **370 GB de RAM.**



---


In [ ]:

# Definimos la cantidad de elementos
N = 10000000

# Creamos los objetos individualmente para que no haya 'interning' (reuso de memoria)
list_floats = [float(i) for i in range(N)]
list_ints = [int(i) for i in range(N)]

def get_full_size(obj):
    # Sumamos el tamaño de la lista + el tamaño de cada objeto dentro
    size_container = sys.getsizeof(obj)
    size_content = sum(sys.getsizeof(item) for item in obj)
    return size_container, size_content

size_cont_f, size_item_f = get_full_size(list_floats)
size_cont_i, size_item_i = get_full_size(list_ints)

print(f"--- ANÁLISIS PARA {N} ELEMENTOS ---")
print(f"FLOATS: Contenedor {size_cont_f}B + Contenido {size_item_f}B = Total {size_cont_f + size_item_f}B")
print(f"INTS:   Contenedor {size_cont_i}B + Contenido {size_item_i}B = Total {size_cont_i + size_item_i}B")

# Verificando la Overallocation (El contenedor debe medir igual)
print(f"\n¿Miden lo mismo los contenedores?: {size_cont_f == size_cont_i}")
print(f"Diferencia real por usar Ints en lugar de Floats: {size_item_i - size_item_f} bytes")

--- ANÁLISIS PARA 10000000 ELEMENTOS ---
FLOATS: Contenedor 89095160B + Contenido 240000000B = Total 329095160B
INTS:   Contenedor 89095160B + Contenido 280000000B = Total 369095160B

¿Miden lo mismo los contenedores?: True
Diferencia real por usar Ints en lugar de Floats: 40000000 bytes



**La RAM es cara y limitada.**
> Cuando llegamos a la escala global de **10,000 millones de tickets**, la diferencia entre usar un `int` y un `float` es de **40 GB de RAM**. ¡Solo por elegir mal el tipo de dato, tendrías que comprar 5 computadoras de 8GB adicionales!
>
> Y como ninguna computadora normal tiene **369 GB** de RAM, o aprendemos a distribuir estos 369 GB en 50 máquinas pequeñas, o StatBurgers quiebra hoy mismo."


# Parte2: Introducción a CLUSTERS (La Brigada)

### 👨‍🍳 El Chef ya no está solo
Como vimos, intentar meter **369 GB** de pedidos en una sola laptop es imposible. El Chef de StatBurgers ha tomado una decisión: **No comprará una supercomputadora.** En su lugar, va a contratar a **4 Ayudantes de Cocina** (Nodos) y les dará a cada uno una mesa pequeña y barata (8 GB de RAM).

### 💡 Concepto Clave: ¿Qué es un Cluster?
Un **Cluster** es un grupo de computadoras (llamadas **Nodos**) que se conectan entre sí para trabajar como si fueran una sola gran máquina.

*   **Escalamiento Vertical:** Comprar una laptop más cara (El Chef solito con una mesa gigante). ❌ *Llega a un tope físico y es muy costoso.*
*   **Escalamiento Horizontal:** Comprar muchas laptops baratas (La Brigada). ✅ *Es infinito: si necesitas más potencia, solo agregas otro ayudante.*

---


### Simular nuestro primer Cluster

En Python, vamos a simular que tenemos 4 Nodos. Cada uno recibirá una parte de los datos que antes hacían colapsar al Chef.

In [ ]:
import numpy as np

# Simulamos la capacidad de 4 Nodos (nuestras 4 mesas de trabajo)
# Cada nodo procesará 2.5 millones de tickets (para completar los 10 millones)

nodo_1 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Bogotá
nodo_2 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Tokio
nodo_3 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Nueva York
nodo_4 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Madrid

print("--- ESTADO DEL CLUSTER ---")
print(f"Nodo 1 (Bogotá):  {nodo_1.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 2 (Tokio):   {nodo_2.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 3 (NY):      {nodo_3.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 4 (Madrid):  {nodo_4.nbytes / 1e6:.2f} MB en uso")

total_ram = (nodo_1.nbytes + nodo_2.nbytes + nodo_3.nbytes + nodo_4.nbytes) / 1e6
print(f"\n✅ Total de datos procesados por la brigada: {total_ram:.2f} MB")

--- ESTADO DEL CLUSTER ---
Nodo 1 (Bogotá):  20.00 MB en uso
Nodo 2 (Tokio):   20.00 MB en uso
Nodo 3 (NY):      20.00 MB en uso
Nodo 4 (Madrid):  20.00 MB en uso

✅ Total de datos procesados por la brigada: 80.00 MB



1.  **¿Dónde están los datos ahora?**

2.  **¿Siguen en un solo lugar?**
---

### 🎯 La Intuición del Chef

Si el Chef quiere saber el total de ventas del día, ya no tiene que contar 10 millones de tickets él solo. Ahora puede gritar: **'¡Equipo, cada uno dígame cuánto sumó su mesa!'**.

Cada ayudante hace su trabajo en paralelo. Mientras el Ayudante 1 cuenta en Bogotá, el Ayudante 2 cuenta en Tokio al mismo tiempo. **Hemos pasado de procesar en serie a procesar en paralelo.**"



# 🔪 PARTE 3 — FRAGMENTACIÓN (Sharding)

### 👨‍🍳 El Chef divide el trabajo
El Chef ya tiene a sus 4 ayudantes (Nodos). Ahora surge un problema logístico: si llegan 10,000 tickets, ¿cómo los repartimos?
No podemos darle los 10,000 a todos (volveríamos a romper las mesas). Tenemos que **romper** el conjunto de datos en pedazos más pequeños.

A este proceso de "partir" la base de datos se le llama **SHARDING** (del inglés *shard*, que significa "fragmento de vidrio roto"). Cada ayudante recibirá un **Shard** (un fragmento) de la información total.

---

### 💡 ¿Cómo funciona el Sharding?
Imagina que tenemos 10,000 clientes numerados del 0 al 9,999. El Chef decide una regla simple de división por rangos:

*   **Nodo 1:** Clientes del 0 al 2,499.
*   **Nodo 2:** Clientes del 2,500 al 4,999.
*   **Nodo 3:** Clientes del 5,000 al 7,499.
*   **Nodo 4:** Clientes del 7,500 al 9,999.

**Cada nodo es "dueño" de una parte de la verdad, pero nadie tiene la verdad completa.**

---

###  Fragmentar los datos de StatBurgers

Ejecuta esta celda para ver cómo se vería la memoria repartida en tu cocina global:


In [ ]:

import numpy as np

# Creamos una lista con 10,000 IDs de pedidos (del 0 al 9,999)
datos_globales = np.arange(10000)

# El Chef reparte los datos (Sharding por rangos)
# INICIA TU CODIGO AQUI
fragmento_1 = datos_globales[:2500]   # Primeros 2,500
fragmento_2 = datos_globales[2500:5000] # Siguientes 2,500
fragmento_3 = datos_globales[5000:7500] # Siguientes 2,500
fragmento_4 = datos_globales[7500:]     # Últimos 2,500
# TERMINA TU CODIGO AQUI

print(f"📦 Nodo 1 tiene {len(fragmento_1)} tickets. Ejemplo IDs: {fragmento_1[:3]}...{fragmento_1[-1]}")
print(f"📦 Nodo 2 tiene {len(fragmento_2)} tickets. Ejemplo IDs: {fragmento_2[:3]}...{fragmento_2[-1]}")
print(f"📦 Nodo 3 tiene {len(fragmento_3)} tickets. Ejemplo IDs: {fragmento_3[:3]}...{fragmento_3[-1]}")
print(f"📦 Nodo 4 tiene {len(fragmento_4)} tickets. Ejemplo IDs: {fragmento_4[:3]}...{fragmento_4[-1]}")

📦 Nodo 1 tiene 2500 tickets. Ejemplo IDs: [0 1 2]...2499
📦 Nodo 2 tiene 2500 tickets. Ejemplo IDs: [2500 2501 2502]...4999
📦 Nodo 3 tiene 2500 tickets. Ejemplo IDs: [5000 5001 5002]...7499
📦 Nodo 4 tiene 2500 tickets. Ejemplo IDs: [7500 7501 7502]...9999




1.  **¿Cada nodo tiene todos los datos?**

2.  **Imagina que el Nodo 2 (Tokio) sufre un apagón y su computadora se quema. ¿Qué pasa con los datos de los clientes del 2,500 al 4,999?**
   

3.  **¿Podemos recuperar esa información preguntándole a los Nodos 1, 3 o 4?**
    

¡La RAM ya no es un problema! Hemos dividido una carga pesada en 4 partes livianas. StatBurgers puede procesar millones de tickets porque cada ayudante solo se preocupa por su pequeño fragmento.

**PERO**, hemos creado un sistema muy frágil. Si un solo ayudante se enferma (falla el nodo), el restaurante pierde una parte de su memoria para siempre.

En el mundo de los datos, **la eficiencia (Sharding) sin seguridad es un suicidio empresarial.**"




# 🧬 PARTE 4 — REPLICACIÓN (El Seguro de Vida)

### 👨‍🍳 El Chef no se arriesga
El Chef de StatBurgers ha aprendido una lección dolorosa: confiar en una sola copia de los datos es peligroso. Si el Nodo 2 (Tokio) se desconecta, perdemos millones de dólares en registros.

Para solucionar esto, usaremos la **REPLICACIÓN**.

### 💡 ¿Qué es la Replicación?
Es el proceso de crear **copias exactas** de los fragmentos de datos en diferentes nodos.
*   **Sharding (Fragmentación):** Es para que los datos *quepan* en la mesa (Eficiencia).
*   **Replicación:** Es para que los datos *no se pierdan* (Disponibilidad).

---

### El Respaldo de Seguridad

Vamos a simular que el **Fragmento 1** (que pertenece originalmente al Nodo 1) ahora tendrá una copia idéntica en el Nodo 2.

In [ ]:
import numpy as np

# Recordemos que fragmento_1 tiene los primeros 2,500 tickets
# Vamos a crear dos réplicas reales en memoria

# INICIA TU CODIGO AQUI
# .copy() es vital: crea un objeto nuevo e independiente en la RAM
replica_en_nodo_1 = fragmento_1.copy()
replica_en_nodo_2 = fragmento_1.copy()
# TERMINA TU CODIGO AQUI

print(f"✅ Réplica Principal (Nodo 1) creada: {len(replica_en_nodo_1)} elementos.")
print(f"✅ Réplica de Respaldo (Nodo 2) creada: {len(replica_en_nodo_2)} elementos.")

# Comprobemos que son independientes (Si cambio una, la otra no cambia)
replica_en_nodo_1[0] = -999
print(f"\nValor en Nodo 1 modificado: {replica_en_nodo_1[0]}")
print(f"Valor en Nodo 2 (sigue intacto): {replica_en_nodo_2[0]}")

✅ Réplica Principal (Nodo 1) creada: 2500 elementos.
✅ Réplica de Respaldo (Nodo 2) creada: 2500 elementos.

Valor en Nodo 1 modificado: -999
Valor en Nodo 2 (sigue intacto): 0





1.  **¿Qué pasa si el Nodo 1 falla por completo?**
    *    *Respuesta:* ¡No pasa nada! El sistema es **Resiliente**. Simplemente le pedimos al Nodo 2 que nos dé la información. StatBurgers sigue vendiendo. A esto lo llamamos **Alta Disponibilidad**.

2.  **¿Qué problema nuevo aparece con este "seguro de vida"?**
    *   *Respuesta:* **El costo de la RAM.** Si replicamos todo 2 veces, ¡necesitamos el **doble de RAM**! Si lo replicamos 3 veces (estándar de la industria como en Google o Amazon), necesitamos el **triple de RAM**.


Por un lado, queremos **Sharding** para que los datos sean pequeños y manejables. Por otro lado, queremos **Replicación** para no perder el negocio si una máquina falla.

En los sistemas distribuidos, **nada es gratis**. Si quieres seguridad (Replicación), tienes que pagar la factura de la memoria. Si quieres ahorrar memoria, te arriesgas a perderlo todo.

El  trabajo es decidir: **¿Cuántas copias son suficientes para dormir tranquilos sin que la factura de la nube nos quiebre?**"

---


# ⚖️ PARTE 5 — EL SANTO GRIAL: FRAGMENTACIÓN + REPLICACIÓN

### 👨‍🍳 La Estrategia Maestra del Chef
El Chef ha diseñado un sistema donde **ningún ayudante es indispensable**, pero **todos los datos son sagrados**.

En lugar de tener un nodo con Sharding (riesgoso) o un nodo con Replicación (costoso), vamos a mezclarlos. Cada fragmento de nuestra base de datos va a vivir en **dos lugares distintos al mismo tiempo**.

### 💡 ¿Cómo se ve este sistema? (Factor de Replicación: 2)
Imagina que dividimos nuestros 10,000 tickets en 4 fragmentos (**F1, F2, F3, F4**). Los repartiremos así:

| Nodo | Contenido (Fragmentos) |
| :--- | :--- |
| **Nodo 1** | Posee **F1** y una copia de **F2** |
| **Nodo 2** | Posee **F2** y una copia de **F1** |
| **Nodo 3** | Posee **F3** y una copia de **F4** |
| **Nodo 4** | Posee **F4** y una copia de **F3** |

---

### Simulación del Cluster StatBurgers

Vamos a construir este cerebro electrónico usando diccionarios de Python para representar los servidores.


In [ ]:

cluster = {
    "nodo_1_bogota": {"F1": fragmento_1, "F2": fragmento_2},
    "nodo_2_tokio":  {"F1": fragmento_1, "F2": fragmento_2},
    "nodo_3_ny":     {"F3": fragmento_3, "F4": fragmento_4},
    "nodo_4_madrid": {"F3": fragmento_3, "F4": fragmento_4},
}

def consultar_ticket_en_cluster(id_ticket):
    # El Chef busca en qué nodo está el ticket sin importar si uno falló
    for nombre_nodo, fragmentos in cluster.items():
        for nombre_f, datos_f in fragmentos.items():
            if id_ticket in datos_f:
                return f"✅ Ticket {id_ticket} encontrado en el {nombre_nodo} (Fragmento {nombre_f})"
    return "❌ Ticket no encontrado."

# Hagamos una prueba
print(consultar_ticket_en_cluster(1050)) # Debería estar en F1
print(consultar_ticket_en_cluster(6000)) # Debería estar en F3

✅ Ticket 1050 encontrado en el nodo_1_bogota (Fragmento F1)
✅ Ticket 6000 encontrado en el nodo_3_ny (Fragmento F3)



### ¿Qué pasa si "muere" el Nodo 1?

Simulemos un desastre: **El servidor de Bogotá (Nodo 1) se incendia.**


In [ ]:
print("💥 ¡ALERTA! El Nodo 1 (Bogotá) ha colapsado y no responde.")

# Eliminamos el nodo 1 del sistema
del cluster["nodo_1_bogota"]

# El Chef intenta buscar un ticket que estaba en el Nodo 1 (por ejemplo, el ID 500)
print("\nBuscando el ticket 500 después del desastre...")
resultado = consultar_ticket_en_cluster(500)
print(resultado)

💥 ¡ALERTA! El Nodo 1 (Bogotá) ha colapsado y no responde.

Buscando el ticket 500 después del desastre...
✅ Ticket 500 encontrado en el nodo_2_tokio (Fragmento F1)




1.  **¿Se perdieron los datos del fragmento F1?**
    *   👉 *Respuesta:* **NO.** Aunque el Nodo 1 desapareció, el **Nodo 2** todavía tiene una copia idéntica de F1. El restaurante sigue operando sin que el cliente se dé cuenta.
2.  **¿Qué pasa con la carga de trabajo?**
    *   👉 *Respuesta:* Aquí hay un detalle importante. Ahora el **Nodo 2** tiene que responder por todas las consultas de F1 y F2 él solo. El sistema es resiliente, pero el Nodo 2 trabajará el doble mientras reparamos el Nodo 1.
3.  **¿Es este sistema perfecto?**
    *   👉 *Respuesta:* Es muy robusto. Para perder datos, tendrían que fallar **el Nodo 1 y el Nodo 2 al mismo tiempo**. La probabilidad de que dos servidores en ciudades distintas fallen en el mismo segundo es bajísima.

---

**Tolerancia a Fallos**.

*   **Sharding** nos dio la capacidad de manejar millones de datos que no cabían en una sola mesa.
*   **Replicación** nos dio la seguridad de que el negocio no muera.

En las grandes empresas como Google o Amazon, no se preguntan *'¿Y si un servidor falla?'*. Ellos asumen que **cientos de servidores van a fallar cada día**. Su software está diseñado para que, cuando una máquina muere, otra tome su lugar automáticamente.

**La clave no es evitar el error, es sobrevivir al error.**"

---



# 🔥 PARTE 6 — VOLVER AL CHEF: MAPREDUCE (Procesar sin mover datos)

### 👨‍🍳 El problema del movimiento de datos
Imagina que tienes **330 GB** de tickets repartidos en tu cluster. Si quieres calcular el total, tienes dos opciones:
1. **La Mala:** Pedirle a todos los nodos que te envíen sus 330 GB por internet a tu laptop para sumarlos. (Tu internet colapsaría y tu RAM explotaría de nuevo).
2. **La Buena (MapReduce):** Enviarles una orden pequeña: *"Calculen la suma en su propia mesa y solo envíenme el resultado final"* (Unos cuantos bytes).

A este paradigma se le llama **MapReduce**:
*   **MAP:** Cada nodo procesa sus datos locales y genera un resultado parcial.
*   **REDUCE:** El Chef recibe los resultados parciales y los combina en una sola respuesta.

---

### El MAX Distribuido (Fácil)
El Gerente quiere saber cuál fue la hamburguesa más cara vendida en el mundo. Esta métrica es "amigable" con la distribución.


In [ ]:
import numpy as np

# MAP: Cada ayudante busca el máximo en su propio fragmento (Mesa local)
max_1 = np.max(fragmento_1)
max_2 = np.max(fragmento_2)
max_3 = np.max(fragmento_3)
max_4 = np.max(fragmento_4)

# REDUCE: El Chef recibe 4 números y elige el mayor de ellos
maximos_locales = [max_1, max_2, max_3, max_4]
max_global = max(maximos_locales)

print(f"🏆 El ticket más caro encontrado por la brigada fue: ${max_global}")

🏆 El ticket más caro encontrado por la brigada fue: $9999



**Veredicto:** El máximo global es igual al máximo de los máximos locales. ¡Todo perfecto!

---

### 💣 Ejercicio 6 — LA TRAMPA: El Promedio de Promedios (¡Peligro!)
El Gerente ahora pide el **Promedio Global de Ventas**. El Chef, confiado por el éxito del máximo, decide pedirle a cada ayudante su promedio local.


In [ ]:
# MAP: Cada ayudante calcula su promedio local
prom_1 = np.mean(fragmento_1)
prom_2 = np.mean(fragmento_2)
prom_3 = np.mean(fragmento_3)
prom_4 = np.mean(fragmento_4)

# REDUCE (El error del Chef): Promediar los promedios
promedios_locales = [prom_1, prom_2, prom_3, prom_4]
promedio_mal = np.mean(promedios_locales)

# --- LA CRUDA REALIDAD ---
# Simulamos el promedio real uniendo todos los datos (fuerza bruta)
todos_los_datos = np.concatenate([fragmento_1, fragmento_2, fragmento_3, fragmento_4])
promedio_real = np.mean(todos_los_datos)

print(f"❌ Cálculo rápido del Chef (Promedio de promedios): {promedio_mal:.4f}")
print(f"✅ Realidad absoluta (Promedio global real):     {promedio_real:.4f}")
print(f"⚠️ Diferencia: {abs(promedio_mal - promedio_real):.4f}")

❌ Cálculo rápido del Chef (Promedio de promedios): 4999.5000
✅ Realidad absoluta (Promedio global real):     4999.5000
⚠️ Diferencia: 0.0000



### ¿Por qué esto está mal?

Dile esto a tus estudiantes: **"Si los números en las mesas no son iguales en cantidad, el promedio de promedios es una mentira."**

**La explicación detallada (Intuición de Andrew Ng):**
Imagina dos mesas:
*   **Mesa A:** 1 cliente que gastó **$100**. (Promedio local = $100).
*   **Mesa B:** 99 clientes que gastaron **$1** cada uno. (Promedio local = $1).

Si el Chef hace: `(100 + 1) / 2 = $50.5`.
El Chef le dirá al Gerente: *"En promedio, cada cliente gasta $50 dólares"*.

**¡MENTIRA!** En realidad hubo 100 clientes y se ganaron $199. El promedio real es **$1.99**.
El Chef se equivocó por **$48.5 dólares por ticket**. En 10 mil millones de ventas, ese error destruiría la multinacional.

---

### 🧠 La Solución Sabia (MapReduce Correcto)

Para calcular el promedio correctamente en un sistema distribuido, el Chef no debe pedir promedios. Debe pedir los **"ingredientes básicos"**.


In [ ]:
# MAP: Cada ayudante envía DOS datos: Suma y Conteo
suma_1, count_1 = np.sum(fragmento_1), len(fragmento_1)
suma_2, count_2 = np.sum(fragmento_2), len(fragmento_2)
suma_3, count_3 = np.sum(fragmento_3), len(fragmento_3)
suma_4, count_4 = np.sum(fragmento_4), len(fragmento_4)

# REDUCE: El Chef suma todas las sumas y divide por todos los conteos
suma_total = suma_1 + suma_2 + suma_3 + suma_4
conteo_total = count_1 + count_2 + count_3 + count_4

promedio_correcto = suma_total / conteo_total

print(f"🎯 Promedio Distribuido Correcto: {promedio_correcto:.4f}")
print(f"✅ Realidad absoluta:             {promedio_real:.4f}")

🎯 Promedio Distribuido Correcto: 4999.5000
✅ Realidad absoluta:             4999.5000



 **No todas las operaciones matemáticas se pueden distribuir de la misma forma.**
*   El **Máximo** es fácil de distribuir.
*   El **Promedio** requiere enviar más 'ingredientes' (Suma y Conteo).


## 🧠  EL PROMEDIO CORRECTO (MapReduce)

Como vimos, el Chef no puede ser perezoso. Para obtener la verdad, debe pedir los ingredientes base.


In [ ]:
suma_total = (
    np.sum(fragmento_1) +
    np.sum(fragmento_2) +
    np.sum(fragmento_3) +
    np.sum(fragmento_4)
)

conteo_total = (
    len(fragmento_1) +
    len(fragmento_2) +
    len(fragmento_3) +
    len(fragmento_4)
)

# REDUCE: El Chef combina todo
promedio_correcto = suma_total / conteo_total

print(f"✅ Promedio Distribuido Sabio: {promedio_correcto:.4f}")
print(f"🎯 Promedio Real (Fuerza bruta): {promedio_real:.4f}")

✅ Promedio Distribuido Sabio: 4999.5000
🎯 Promedio Real (Fuerza bruta): 4999.5000



**Lección Técnica:** El promedio es una métrica **algebraica**. Se puede descomponer en funciones más simples (suma y conteo) que sí son distribuibles.

---

# 💀 PARTE 7 — EL PROBLEMA REAL: LA MEDIANA

El Gerente de StatBurgers vuelve a la cocina con una última petición:
> *"Chef, el promedio está sesgado por unos pocos clientes VIP que gastan miles de dólares. Necesito la **MEDIANA EXACTA**. Quiero saber cuánto gasta el cliente 'de la mitad'."*

Aquí es donde el sistema distribuido se enfrenta a su mayor reto.

### 🧪 Ejercicio

Intentemos aplicar la lógica de los ayudantes a la mediana:


In [ ]:
# MAP: Cada ayudante calcula su mediana local
mediana_1 = np.median(fragmento_1)
mediana_2 = np.median(fragmento_2)
mediana_3 = np.median(fragmento_3)
mediana_4 = np.median(fragmento_4)

# El Chef intenta promediar las medianas (¿Funcionará?)
mediana_ingenuo = np.mean([mediana_1, mediana_2, mediana_3, mediana_4])

# La Verdad Absoluta
mediana_real = np.median(todos_los_datos)

print(f"❌ Mediana 'Distribuida' (Promedio de medianas): {mediana_ingenuo:.4f}")
print(f"🎯 Mediana Real:                               {mediana_real:.4f}")

❌ Mediana 'Distribuida' (Promedio de medianas): 4999.5000
🎯 Mediana Real:                               4999.5000



### ❓ Preguntas para Discusión Obligatoria:

1.  **¿Puedo promediar estas medianas locales?**
    *   👉 *Respuesta:* **NO.** La mediana es una métrica **holística**. Depende del orden de TODOS los datos. Si en el Nodo 1 todos los datos son bajos (1, 2, 3) y en el Nodo 2 todos son altos (100, 200, 300), promediar sus medianas (2 y 200) no te dirá nada sobre el centro de la distribución global.

2.  **¿Puedo juntar todos los datos (330 GB) en mi mesa para ordenarlos y sacar la mediana?**
    *   👉 *Respuesta:* **NO.** Recuerda la Parte 1: Tu RAM explotará. El `sort` (ordenamiento) es una de las operaciones que más memoria consume.

---



Hemos llegado al límite de lo que el MapReduce simple puede hacer. La mediana requiere saber el orden de cada ticket respecto a todos los demás del mundo.

Si no podemos mover los datos a una sola máquina y no podemos promediar resultados locales... **¿Cómo lo resuelven Google o Amazon?**"

**Soluciones propuestas (El debate):**

*   **Aproximación por Histogramas:** Cada ayudante no envía la mediana, envía un **Histograma** (cuántos tickets tiene entre $0-10, $10-20, etc.). El Chef suma los histogramas y así sabe en qué rango "cae" la mediana. Luego les pide a los ayudantes solo los datos de ese rango.
*   **Algoritmos Iterativos:** El Chef propone una cifra (ej. $20) y los ayudantes responden: *"Tengo X tickets por debajo de esa cifra"*. El Chef ajusta su propuesta hasta encontrar el centro.
*   **Muestreo:** Tomar una muestra aleatoria pequeña de cada nodo y calcular la mediana de la muestra (asumiendo un margen de error).

---



| Concepto | Problema que resuelve | Analogía del Chef |
| :--- | :--- | :--- |
| **Cluster** | Escalar el cómputo y la memoria. | Contratar una brigada de 100 cocineros. |
| **Fragmentación (Sharding)** | Dividir datos que no caben en una mesa. | Partir la lista de pedidos en 4 mesas. |
| **Replicación** | Tolerancia a fallos (Seguridad). | Tener una copia de cada pedido en otra mesa. |
| **MapReduce** | Procesar datos sin moverlos (Eficiencia). | Que cada cocinero cuente sus tickets y te dé el total. |

---



# ⚖️ PARTE 8 — TRADE-OFFS: Decisiones

### 🧠 El principio de "Nada es gratis" (No Free Lunch)
En el mundo de los datos distribuidos, el Chef ya no solo cocina; ahora administra recursos. Cada decisión técnica es una balanza:

| Decisión | Beneficio (Lo que ganas) | Costo (Lo que sacrificas) |
| :--- | :--- | :--- |
| **Sharding** | **Escala:** Puedes guardar datos infinitos. | **Complejidad:** Es más difícil saber dónde está cada dato. |
| **Replicación** | **Seguridad:** El sistema nunca muere. | **Factura:** Pagas el doble o triple de RAM. |
| **Cluster** | **Velocidad:** Muchos procesan al tiempo. | **Coordinación:** Alguien debe organizar a los nodos. |
| **Aproximación** | **Inmediatez:** Tienes resultados ya mismo. | **Precisión:** No tienes la cifra exacta (ej. Mediana). |

# 🧪 💣 PARTE 9 — EJERCICIO FINAL

### 🎯 El Problema de la Realidad Desproporcionada
Llegó el final del día en la multinacional. Tienes 3 sucursales con volúmenes de venta totalmente distintos. Si cometes el error de "Chef de barrio", reportarás métricas falsas que destruirán el negocio.

**Configuración del Cluster:**
*   **Nodo A (Sucursal VIP):** 100 pedidos (Venta promedio = $100 USD)
*   **Nodo B (Sucursal Ciudad):** 10,000 pedidos (Venta promedio = $10 USD)
*   **Nodo C (Sucursal Aeropuerto):** 1,000,000 pedidos (Venta promedio = $1 USD)


In [ ]:
# --- CONFIGURACIÓN DE LOS NODOS ---
n_a, prom_a = 100, 100
n_b, prom_b = 10000, 10
n_c, prom_c = 1000000, 1

# ❓ PREGUNTA 1: Calcula el promedio usando el "Promedio de Promedios" (Método Incorrecto)
# INICIA TU CODIGO AQUI
promedio_de_promedios = (prom_a + prom_b + prom_c) / 3
# TERMINA TU CODIGO AQUI

# ❓ PREGUNTA 2: Calcula el promedio global correcto (Método Distribuido/Ponderado)
# Pista: Debes reconstruir la Suma Total Global y dividirla por el Conteo Total Global
# INICIA TU CODIGO AQUI
suma_total = (n_a * prom_a) + (n_b * prom_b) + (n_c * prom_c)
conteo_total = n_a + n_b + n_c
promedio_real = suma_total / conteo_total
# TERMINA TU CODIGO AQUI

print(f"❌ Promedio 'Mentirosito': ${promedio_de_promedios:.2f}")
print(f"✅ Promedio Real Sabio:   ${promedio_real:.2f}")

❌ Promedio 'Mentirosito': $37.00
✅ Promedio Real Sabio:   $1.10



### 🧠 Reflexión final del ejercicio:
1.  **¿Por qué falló el primer cálculo?** Porque le dio el mismo "peso" a 100 personas que a un millón.
2.  **¿Cuál nodo domina el resultado?** El **Nodo C**. Al ser masivo, su realidad (promedio de $1) "arrastra" la estadística global. En Big Data, el volumen es el que manda.

--

# PARTE 10: Arquitectura del Ecosistema de Datos Distribuidos

Cuando pasamos de una sola computadora a un entorno empresarial, dejamos de hablar de "listas de Python" y empezamos a hablar de **Sistemas Distribuidos**. Estas herramientas están diseñadas para manejar los tres retos del Big Data: **Volumen, Velocidad y Variedad.**

---

## 1. Capa de Almacenamiento (Storage): El Cimiento del Cluster

El primer problema es: *¿Dónde guardamos 500 Terabytes si el disco duro más grande tiene 20 TB?* La respuesta es el **Almacenamiento Distribuido**.

### A. Apache Hadoop HDFS (Hadoop Distributed File System)
Es el estándar histórico. Su magia reside en dos conceptos:
*   **Bloques:** HDFS toma un archivo de 1 TB y lo "pica" en trozos de **128 MB**.
*   **Replicación de Bloques:** Cada trozo de 128 MB se copia, por defecto, **3 veces** en servidores diferentes (Nodos).
*   **¿Por qué?:** Si un servidor falla, el sistema ni siquiera te avisa; simplemente busca el bloque en la copia 2 o 3. Es el concepto de **Tolerancia a Fallos** llevado al extremo.

### B. Amazon S3 / Google Cloud Storage (Object Storage)
A diferencia de un disco duro normal, estos servicios guardan "objetos". Son la base de lo que llamamos **Data Lakes**.
*   **Escalabilidad Infinita:** No tienes que preocuparte por el Sharding; el proveedor de nube lo hace por debajo.
*   **Durabilidad del 99.999999999%:** Están diseñados para que la probabilidad de perder un archivo sea casi cero, replicando los datos incluso en diferentes regiones geográficas (ej. una copia en Virginia y otra en Ohio).


---

## 2. Capa de Procesamiento (Compute): El Cerebro Distribuido

Una vez que los datos están repartidos, necesitamos procesarlos. Aquí aplicamos el paradigma de **MapReduce**: "Llevar el código a los datos, no los datos al código".

### A. Apache Spark (El Estándar de la Industria)
Spark revolucionó el Big Data porque procesa casi todo **en RAM**, lo que lo hace 100 veces más rápido que el MapReduce tradicional.
*   **RDD (Resilient Distributed Datasets):** Es la abstracción de una lista de Python, pero distribuida. Si una parte de la lista se pierde (porque un nodo falló), Spark conoce el "linaje" (la receta) y puede reconstruir solo ese pedazo sin reiniciar todo el proceso.
*   **Lazy Evaluation:** Spark no ejecuta nada hasta que le pides un resultado final. Primero crea un plan optimizado para minimizar el movimiento de datos entre nodos (Shuffle).

### B. Dask
Es la respuesta para la comunidad de Python.
*   **Pandas a Escala:** Si una tabla de Pandas de 50 GB no cabe en tu RAM, Dask la divide en 50 tablas de 1 GB y las procesa en paralelo usando todos los núcleos de tu procesador o múltiples computadoras.


---

## 3. Data Warehousing Moderno: La "Cocina Invisible"

En estas herramientas, la complejidad distribuida está "escondida". Tú solo escribes SQL y la herramienta hace el Sharding y MapReduce por ti.

*   **Google BigQuery:** Utiliza una tecnología llamada *Dremel*. Separa totalmente el disco duro del procesador. Cuando haces un `SELECT SUM(ventas)`, BigQuery asigna miles de ranuras (slots) de procesamiento que leen los datos en paralelo y reducen el resultado en segundos.
*   **Snowflake:** Su gran innovación es la **Elasticidad**. Puedes tener 10 nodos trabajando al mediodía cuando hay mucho tráfico, y reducirlos a 1 nodo a la medianoche para ahorrar dinero, sin mover los datos de lugar.

---

## 4. Orquestación: Los Directores de la Orquesta

Un sistema distribuido tiene muchas partes móviles. Necesitamos a alguien que vigile la salud del cluster.

*   **Kubernetes (K8s):** Es el sistema operativo de la nube. Si un nodo que está procesando datos se muere, Kubernetes lanza un reemplazo en milisegundos. Asegura que la **Replicación** que definiste se mantenga siempre activa.
*   **Hadoop YARN:** Es el "negociador" de recursos. Si Spark pide 100 GB de RAM para un cálculo, YARN revisa qué nodos tienen espacio libre y les asigna la tarea.

---

## 💡 Resumen de Conexión Técnica

| Concepto del Taller | Realidad en la Industria | Herramienta Ejemplo |
| :--- | :--- | :--- |
| **RAM Insuficiente** | **Particionamiento** | **Spark** divide los datos en pedazos que caben en la RAM de cada nodo. |
| **Sharding** | **Distribución de Carga** | **Cassandra** reparte usuarios por todo el mundo para que ningún servidor se sature. |
| **Replicación** | **Alta Disponibilidad** | **HDFS/S3** aseguran que si un centro de datos se inunda, tus datos sigan vivos en otro. |
| **MapReduce** | **Paralelismo Masivo** | **BigQuery** suma billones de filas usando miles de CPUs simultáneamente. |

---


Como Científico de Datos o Ingeniero de ML, tu valor no está en saber instalar estas herramientas, sino en **entender la lógica distribuida** que hay detrás.

Si entiendes que un `count()` es fácil de distribuir pero una `mediana()` es costosa, sabrás por qué tu consulta en **BigQuery** tarda más o por qué tu modelo en **Spark** necesita más nodos. El paradigma distribuido es lo que permite que empresas como **Rappi, Nubank o Google** manejen millones de usuarios sin que sus sistemas colapsen.

**Dominar el pensamiento distribuido es lo que te permite escalar de "analista de hojas de cálculo" a "arquitecto de soluciones de impacto global".**

# Bibliografía recomendada
## Bibliografía base
1. Tanenbaum, A. S., & van Steen, M. *Distributed Systems: Principles and Paradigms*.
2. Kleppmann, M. *Designing Data-Intensive Applications*.
3. White, T. *Hadoop: The Definitive Guide*.
## Lecturas complementarias
1. Dean, J., & Ghemawat, S. (2004). *MapReduce: Simplified Data Processing on Large Clusters*.
2. Dean, J., & Ghemawat, S. (2008). *MapReduce: simplified data processing on large clusters*. Communications of the ACM.
3. Ghemawat, S., Gobioff, H., & Leung, S.-T. (2003). *The Google File System*.
4. Chang, F. et al. (2008). *Bigtable: A Distributed Storage System for Structured Data*.